# 03 - Results Comparison

This notebook compares baseline and self-training results using statistical analysis and publication-quality figures. It works with mock data now and can be re-run with real results after training completes.

**What you will learn:**
1. How to load and compare per-subject metrics
2. Statistical significance testing (Wilcoxon signed-rank)
3. Per-class analysis with box plots
4. Convergence and improvement visualization
5. How to interpret the results

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

%matplotlib inline

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

rng = np.random.RandomState(42)

## 1. Generate Mock Results

These mock results simulate realistic improvements from self-training. Replace this cell with actual data loading when results are available.

**To use real data**, replace the mock data generation with:
```python
baseline_df = pd.read_csv("../results/baseline_metrics.csv")
round_dfs = {r: pd.read_csv(f"../results/round{r}_metrics.csv") for r in range(4)}
```

In [ ]:
n_subjects = 97
class_names = ["TC", "WT", "ET"]

# Simulate realistic per-subject Dice distributions
# Baseline: approximately TC=0.84, WT=0.91, ET=0.82
def generate_dice_scores(tc_mean=0.84, wt_mean=0.91, et_mean=0.82, n=97, seed=42):
    rng_local = np.random.RandomState(seed)
    tc = np.clip(rng_local.normal(tc_mean, 0.08, n), 0.3, 1.0)
    wt = np.clip(rng_local.normal(wt_mean, 0.05, n), 0.5, 1.0)
    et = np.clip(rng_local.normal(et_mean, 0.10, n), 0.2, 1.0)
    return pd.DataFrame({"TC": tc, "WT": wt, "ET": et, "Mean": (tc + wt + et) / 3})


# Generate mock data for baseline and each round
baseline_df = generate_dice_scores(0.84, 0.91, 0.82, seed=42)
round_dfs = {
    0: generate_dice_scores(0.855, 0.915, 0.835, seed=43),
    1: generate_dice_scores(0.865, 0.920, 0.845, seed=44),
    2: generate_dice_scores(0.870, 0.922, 0.852, seed=45),
    3: generate_dice_scores(0.872, 0.923, 0.855, seed=46),
}

print("Mock data generated for:")
print(f"  Baseline:  {n_subjects} subjects")
for r in range(4):
    print(f"  Round {r}:   {n_subjects} subjects")

## 2. Summary Metrics Table

In [ ]:
# Build summary table
summary_rows = []
summary_rows.append(
    {
        "Round": "Baseline",
        "TC": f"{baseline_df['TC'].mean():.4f}",
        "WT": f"{baseline_df['WT'].mean():.4f}",
        "ET": f"{baseline_df['ET'].mean():.4f}",
        "Mean": f"{baseline_df['Mean'].mean():.4f}",
        "Delta": "---",
    }
)

for r in range(4):
    delta = round_dfs[r]["Mean"].mean() - baseline_df["Mean"].mean()
    summary_rows.append(
        {
            "Round": f"Round {r}",
            "TC": f"{round_dfs[r]['TC'].mean():.4f}",
            "WT": f"{round_dfs[r]['WT'].mean():.4f}",
            "ET": f"{round_dfs[r]['ET'].mean():.4f}",
            "Mean": f"{round_dfs[r]['Mean'].mean():.4f}",
            "Delta": f"+{delta:.4f}",
        }
    )

summary_df = pd.DataFrame(summary_rows)
print("Per-Round Mean Dice Scores:")
print(summary_df.to_string(index=False))

## 3. Per-Round Improvement Bar Chart

In [ ]:
rounds_labels = ["Baseline", "Round 0", "Round 1", "Round 2", "Round 3"]
all_dfs = [baseline_df] + [round_dfs[r] for r in range(4)]

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(rounds_labels))
width = 0.2
class_colors = {"TC": "#e74c3c", "WT": "#27ae60", "ET": "#f39c12"}

for i, cls in enumerate(class_names):
    means = [df[cls].mean() for df in all_dfs]
    stds = [df[cls].std() for df in all_dfs]
    ax.bar(
        x + i * width - width,
        means,
        width,
        yerr=stds,
        label=cls,
        color=class_colors[cls],
        alpha=0.8,
        capsize=3,
    )

ax.set_xlabel("Training Stage", fontsize=12)
ax.set_ylabel("Dice Score", fontsize=12)
ax.set_title("Per-Class Dice Scores: Baseline vs Self-Training Rounds", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(rounds_labels)
ax.legend(fontsize=10)
ax.set_ylim(0.75, 1.0)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4. Statistical Significance Testing

We use the **paired Wilcoxon signed-rank test** to determine whether self-training improvements are statistically significant. This is a non-parametric test appropriate for paired samples (same subjects, different methods).

In [ ]:
print("Paired Wilcoxon Signed-Rank Tests (vs Baseline):")
print("=" * 70)
print(f"{'Comparison':<25} {'Metric':<8} {'Mean Diff':>10} {'p-value':>12} {'Sig?':>6}")
print("-" * 70)

for r in range(4):
    for cls in ["Mean"] + class_names:
        diff = round_dfs[r][cls].values - baseline_df[cls].values
        stat, p_val = stats.wilcoxon(diff, alternative="greater")
        sig = "Yes" if p_val < 0.05 else "No"
        print(f"Round {r} vs Baseline    {cls:<8} {diff.mean():>+10.4f} {p_val:>12.2e} {sig:>6}")
    print("-" * 70)

print()
print("Interpretation:")
print("  p < 0.05: The improvement is statistically significant.")
print("  We use a one-sided test (alternative='greater') because we")
print("  hypothesize that self-training improves, not just changes, Dice.")

## 5. Box Plot Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, cls in enumerate(class_names):
    data = [baseline_df[cls].values] + [round_dfs[r][cls].values for r in range(4)]
    bp = axes[i].boxplot(
        data,
        labels=rounds_labels,
        patch_artist=True,
        widths=0.6,
        showmeans=True,
        meanprops={"marker": "D", "markerfacecolor": "black", "markersize": 6},
    )

    colors_bp = ["#bdc3c7"] + ["#85c1e9", "#76d7c4", "#f9e79f", "#f5b7b1"]
    for patch, color in zip(bp["boxes"], colors_bp):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    axes[i].set_ylabel("Dice Score")
    axes[i].set_title(f"{cls} Dice Distribution", fontsize=12, fontweight="bold")
    axes[i].grid(True, alpha=0.3, axis="y")

    # Add significance markers
    _, p_val = stats.wilcoxon(
        round_dfs[3][cls].values - baseline_df[cls].values, alternative="greater"
    )
    if p_val < 0.001:
        sig_text = "***"
    elif p_val < 0.01:
        sig_text = "**"
    elif p_val < 0.05:
        sig_text = "*"
    else:
        sig_text = "ns"

    y_max = max([d.max() for d in data]) + 0.02
    axes[i].plot([1, 5], [y_max, y_max], "k-", linewidth=1)
    axes[i].text(3, y_max + 0.005, sig_text, ha="center", fontsize=12, fontweight="bold")

fig.suptitle(
    "Per-Class Dice Score Distributions: Baseline vs Self-Training",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

print("Diamond markers = means | Lines = medians | * p<0.05, ** p<0.01, *** p<0.001")

## 6. Per-Subject Improvement Analysis

Not all subjects benefit equally from self-training. Let's analyze where the improvements (and regressions) occur.

In [ ]:
best_round = 3
diff_mean = round_dfs[best_round]["Mean"].values - baseline_df["Mean"].values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of per-subject improvements
ax1.hist(diff_mean, bins=30, color="#3498db", alpha=0.7, edgecolor="black", linewidth=0.5)
ax1.axvline(x=0, color="red", linestyle="--", linewidth=2, label="No change")
ax1.axvline(x=diff_mean.mean(), color="black", linestyle="-", linewidth=2, label=f"Mean: {diff_mean.mean():+.4f}")
ax1.set_xlabel("Dice Improvement (Round 3 - Baseline)")
ax1.set_ylabel("Number of Subjects")
ax1.set_title("Distribution of Per-Subject Improvements", fontsize=12, fontweight="bold")
ax1.legend()

improved = (diff_mean > 0).sum()
degraded = (diff_mean < 0).sum()
unchanged = (diff_mean == 0).sum()
ax1.text(
    0.02,
    0.95,
    f"Improved: {improved}/{n_subjects}\nDegraded: {degraded}/{n_subjects}",
    transform=ax1.transAxes,
    fontsize=10,
    verticalalignment="top",
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.8},
)

# Scatter: baseline vs self-trained
ax2.scatter(
    baseline_df["Mean"].values,
    round_dfs[best_round]["Mean"].values,
    alpha=0.5,
    s=30,
    c="#3498db",
    edgecolors="black",
    linewidth=0.3,
)
lims = [0.5, 1.0]
ax2.plot(lims, lims, "r--", linewidth=1.5, label="y=x (no improvement)")
ax2.set_xlabel("Baseline Mean Dice")
ax2.set_ylabel("Self-Trained Mean Dice (Round 3)")
ax2.set_title("Baseline vs Self-Trained (Per Subject)", fontsize=12, fontweight="bold")
ax2.legend()
ax2.set_xlim(lims)
ax2.set_ylim(lims)
ax2.set_aspect("equal")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Subjects improved: {improved}/{n_subjects} ({100*improved/n_subjects:.1f}%)")
print(f"Subjects degraded: {degraded}/{n_subjects} ({100*degraded/n_subjects:.1f}%)")
print(f"Mean improvement:  {diff_mean.mean():+.4f} Dice")
print(f"Max improvement:   {diff_mean.max():+.4f}")
print(f"Max degradation:   {diff_mean.min():+.4f}")

## 7. Convergence Across Rounds

In [ ]:
# Simulate training curves for each round
epochs_per_round = 100

def simulate_training_curve(start_dice, end_dice, epochs=100, noise=0.005):
    """Simulate a Dice score training curve with logarithmic convergence."""
    t = np.linspace(0, 1, epochs)
    # Logarithmic-like convergence
    curve = start_dice + (end_dice - start_dice) * (1 - np.exp(-4 * t))
    # Add noise
    noise_vals = rng.normal(0, noise, epochs)
    return curve + noise_vals


# Simulate mean Dice curves for each round
round_curves = {
    "Baseline": simulate_training_curve(0.50, 0.857, epochs=300, noise=0.008),
    "Round 0": simulate_training_curve(0.857, 0.870, noise=0.004),
    "Round 1": simulate_training_curve(0.870, 0.877, noise=0.003),
    "Round 2": simulate_training_curve(0.877, 0.881, noise=0.003),
    "Round 3": simulate_training_curve(0.881, 0.883, noise=0.002),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Left: Full training history
curve_colors = ["#95a5a6", "#3498db", "#2ecc71", "#f39c12", "#e74c3c"]
offset = 0
for i, (name, curve) in enumerate(round_curves.items()):
    epochs_arr = np.arange(offset, offset + len(curve))
    ax1.plot(epochs_arr, curve, linewidth=1.5, label=name, color=curve_colors[i], alpha=0.8)
    if i > 0:
        ax1.axvline(x=offset, color="gray", linestyle=":", alpha=0.5)
    offset += len(curve)

ax1.set_xlabel("Cumulative Epoch")
ax1.set_ylabel("Validation Mean Dice")
ax1.set_title("Full Training History", fontsize=12, fontweight="bold")
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# Right: Mean Dice at end of each stage
final_dice = [curve[-1] for curve in round_curves.values()]
bar_colors = curve_colors
ax2.bar(range(len(round_curves)), final_dice, color=bar_colors, alpha=0.8, edgecolor="black")
ax2.set_xticks(range(len(round_curves)))
ax2.set_xticklabels(round_curves.keys(), fontsize=9)
ax2.set_ylabel("Final Validation Mean Dice")
ax2.set_title("Best Mean Dice Per Stage", fontsize=12, fontweight="bold")
ax2.set_ylim(0.82, 0.90)
ax2.grid(True, alpha=0.3, axis="y")

for i, val in enumerate(final_dice):
    ax2.text(i, val + 0.001, f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

## 8. Summary and Conclusions

In [ ]:
print("=" * 60)
print("RESULTS SUMMARY (Mock Data)")
print("=" * 60)
print()

baseline_mean = baseline_df["Mean"].mean()
best_mean = round_dfs[3]["Mean"].mean()
improvement = best_mean - baseline_mean

print(f"Baseline Mean Dice:      {baseline_mean:.4f}")
print(f"Best Self-Trained Dice:  {best_mean:.4f} (Round 3)")
print(f"Absolute Improvement:    {improvement:+.4f}")
print(f"Relative Improvement:    {100 * improvement / baseline_mean:+.2f}%")
print()

print("Per-Class Improvements (Round 3 vs Baseline):")
for cls in class_names:
    bl = baseline_df[cls].mean()
    st = round_dfs[3][cls].mean()
    _, p = stats.wilcoxon(round_dfs[3][cls].values - baseline_df[cls].values, alternative="greater")
    print(f"  {cls}: {bl:.4f} -> {st:.4f} ({st - bl:+.4f}, p={p:.2e})")

print()
print("Key Findings:")
print("  1. Self-training consistently improves performance across all rounds")
print("  2. Most improvement occurs in rounds 0-1; diminishing returns after")
print("  3. All three tumor classes benefit, with ET showing the largest gain")
print("  4. Improvements are statistically significant (p < 0.05)")
print()
print("NOTE: These are MOCK results. Replace with actual metrics after training.")

---

**Next notebook:** [04_educational_guide.ipynb](04_educational_guide.ipynb) provides a broader tutorial on semi-supervised learning concepts.